# MLP em PyTorch vs. XGBoost/LightGBM — Projeto Lupa (Módulo 9)

Mesmo target (`houve_glosa`), mesmo split out-of-time, mesmas features do Módulo 7 — comparação justa entre um MLP simples e o LightGBM já promovido no MLflow.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import average_precision_score, roc_auc_score

from src.preprocessing import (
    carregar_dados, construir_pipeline_preprocessamento, criar_target,
    get_engine, split_out_of_time,
)

torch.manual_seed(42)

engine = get_engine()
df = criar_target(carregar_dados(engine))
treino, teste = split_out_of_time(df, ano_corte=2026, mes_corte=4)
treino_fit, treino_val = split_out_of_time(treino, ano_corte=2026, mes_corte=1)

pipeline_prep = construir_pipeline_preprocessamento()
X_fit = pipeline_prep.fit_transform(treino_fit)
X_val = pipeline_prep.transform(treino_val)
X_teste = pipeline_prep.transform(teste)

y_fit = treino_fit["houve_glosa"].values
y_val = treino_val["houve_glosa"].values
y_teste = teste["houve_glosa"].values

def para_tensor(X):
    return torch.tensor(X.toarray() if hasattr(X, "toarray") else X, dtype=torch.float32)

X_fit_t, X_val_t, X_teste_t = para_tensor(X_fit), para_tensor(X_val), para_tensor(X_teste)
y_fit_t = torch.tensor(y_fit, dtype=torch.float32).unsqueeze(1)

print(f"Treino: {X_fit_t.shape} | Val: {X_val_t.shape} | Teste: {X_teste_t.shape}")

Treino: torch.Size([541991, 69]) | Val: torch.Size([43381, 69]) | Teste: torch.Size([55146, 69])


## Definindo e treinando o MLP

Arquitetura simples: 2 camadas escondidas, dropout pra regularizar. `pos_weight` no `BCEWithLogitsLoss` é o equivalente do `scale_pos_weight`/`is_unbalance` do XGBoost/LightGBM — reponderando a classe rara sem precisar de resample.

In [2]:
class MLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.rede = nn.Sequential(
            nn.Linear(n_features, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.rede(x)


peso_classe_positiva = (y_fit == 0).sum() / (y_fit == 1).sum()
modelo_mlp = MLP(n_features=X_fit_t.shape[1])
otimizador = torch.optim.Adam(modelo_mlp.parameters(), lr=1e-3)
funcao_perda = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(peso_classe_positiva, dtype=torch.float32))

N_EPOCAS = 15
TAMANHO_LOTE = 2048
n_amostras = X_fit_t.shape[0]

melhor_pr_auc_val = 0.0
for epoca in range(N_EPOCAS):
    modelo_mlp.train()
    permutacao = torch.randperm(n_amostras)
    perda_epoca = 0.0
    for i in range(0, n_amostras, TAMANHO_LOTE):
        indices_lote = permutacao[i : i + TAMANHO_LOTE]
        X_lote, y_lote = X_fit_t[indices_lote], y_fit_t[indices_lote]

        otimizador.zero_grad()
        saida = modelo_mlp(X_lote)
        perda = funcao_perda(saida, y_lote)
        perda.backward()
        otimizador.step()
        perda_epoca += perda.item() * len(indices_lote)

    modelo_mlp.eval()
    with torch.no_grad():
        score_val = torch.sigmoid(modelo_mlp(X_val_t)).numpy().ravel()
    pr_auc_val = average_precision_score(y_val, score_val)
    melhor_pr_auc_val = max(melhor_pr_auc_val, pr_auc_val)
    print(f"Época {epoca+1}/{N_EPOCAS} - perda: {perda_epoca/n_amostras:.4f} - PR-AUC (val): {pr_auc_val:.4f}")

Época 1/15 - perda: 0.9654 - PR-AUC (val): 0.3291


Época 2/15 - perda: 0.8208 - PR-AUC (val): 0.3451


Época 3/15 - perda: 0.8030 - PR-AUC (val): 0.3629


Época 4/15 - perda: 0.7942 - PR-AUC (val): 0.3750


Época 5/15 - perda: 0.7854 - PR-AUC (val): 0.3833


...

Época 11/15 - perda: 0.7617 - PR-AUC (val): 0.4009


Época 12/15 - perda: 0.7596 - PR-AUC (val): 0.4033


Época 13/15 - perda: 0.7539 - PR-AUC (val): 0.4117


Época 14/15 - perda: 0.7537 - PR-AUC (val): 0.4084


Época 15/15 - perda: 0.7520 - PR-AUC (val): 0.4100


## Avaliação no teste real, e comparação honesta com o LightGBM (Módulo 7)

In [3]:
modelo_mlp.eval()
with torch.no_grad():
    score_teste_mlp = torch.sigmoid(modelo_mlp(X_teste_t)).numpy().ravel()

pr_auc_mlp = average_precision_score(y_teste, score_teste_mlp)
auc_roc_mlp = roc_auc_score(y_teste, score_teste_mlp)

n_params_mlp = sum(p.numel() for p in modelo_mlp.parameters())

print("=== MLP (PyTorch) ===")
print(f"PR-AUC (teste real): {pr_auc_mlp:.4f}")
print(f"AUC-ROC (teste real): {auc_roc_mlp:.4f}")
print(f"Parâmetros treináveis: {n_params_mlp:,}")

print("\n=== LightGBM (Módulo 7, para comparação) ===")
print("PR-AUC (teste real): 0.4121")
print("AUC-ROC (teste real): 0.8938")

=== MLP (PyTorch) ===
PR-AUC (teste real): 0.3493
AUC-ROC (teste real): 0.8805
Parâmetros treináveis: 6,593

=== LightGBM (Módulo 7, para comparação) ===
PR-AUC (teste real): 0.4121
AUC-ROC (teste real): 0.8938
